# Phase 7 — Model Validation & Accuracy Audit

## Objective

This phase audits the current candidate model without changing it. It checks generalization, fraud-class performance, probability behavior, threshold diagnostics, stability, overfitting, errors, and possible leakage.

No hyperparameter tuning, SMOTE, threshold optimization, retraining of the saved candidate, feature selection, SHAP, or deployment is performed.

## 1. Load Data, Candidate Model, and Phase 6 Results

In [ ]:
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import sparse

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, confusion_matrix,
    precision_score, recall_score, f1_score, classification_report,
    roc_auc_score, average_precision_score, precision_recall_curve
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.base import clone

project = Path.cwd().parent
processed = project / "data" / "processed"
models_folder = project / "models"
results_folder = project / "results"
X_train = sparse.load_npz(processed / "X_train_processed.npz")
X_test = sparse.load_npz(processed / "X_test_processed.npz")
y_train = pd.read_csv(processed / "y_train.csv")["isFraud"]
y_test = pd.read_csv(processed / "y_test.csv")["isFraud"]
phase6 = pd.read_csv(results_folder / "model_comparison.csv")
model_files = sorted(p.name for p in models_folder.glob("*.pkl"))
current_candidate = "Random Forest"
candidate = joblib.load(models_folder / "random_forest.pkl")
print("Available model files:", model_files)
print("Current Candidate:", current_candidate)
print(phase6[phase6["Model"] == current_candidate][["PR-AUC", "F1", "Recall", "Precision"]])

## 2. Validate Data and Candidate Selection

In [ ]:
assert X_train.shape[0] == len(y_train)
assert X_test.shape[0] == len(y_test)
assert X_train.shape[1] == X_test.shape[1]
assert np.isfinite(X_train.data).all()
assert np.isfinite(X_test.data).all()
phase6_row = phase6.loc[phase6["Model"] == current_candidate].iloc[0]
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("NaN/Inf audit: passed")
print("Candidate reason: highest Phase 6 PR-AUC and F1 among compared models.")
print("Phase 6 PR-AUC:", phase6_row["PR-AUC"])
print("Phase 6 F1:", phase6_row["F1"])
print("Phase 6 Recall:", phase6_row["Recall"])
print("Phase 6 Precision:", phase6_row["Precision"])

## 3. Regenerate Train/Test Predictions and Probabilities

In [ ]:
y_train_pred = candidate.predict(X_train)
y_test_pred = candidate.predict(X_test)
y_train_proba = candidate.predict_proba(X_train)[:, 1]
y_test_proba = candidate.predict_proba(X_test)[:, 1]
assert len(y_train_pred) == len(y_train)
assert len(y_test_pred) == len(y_test)
assert len(y_train_proba) == len(y_train)
assert len(y_test_proba) == len(y_test)
assert 0 <= y_test_proba.min() <= 1 and 0 <= y_test_proba.max() <= 1
print("Train predictions:", len(y_train_pred))
print("Test predictions:", len(y_test_pred))
print("Train probabilities:", len(y_train_proba))
print("Test probabilities:", len(y_test_proba))
print("Test probability min/max/mean/median:", y_test_proba.min(), y_test_proba.max(), y_test_proba.mean(), np.median(y_test_proba))

## 4. Recalculate Complete Metrics

In [ ]:
def metrics(y_true, pred, proba):
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return {
        "Accuracy": accuracy_score(y_true, pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, proba),
        "PR-AUC": average_precision_score(y_true, proba),
        "TN": tn, "FP": fp, "FN": fn, "TP": tp
    }
train_metrics = metrics(y_train, y_train_pred, y_train_proba)
test_metrics = metrics(y_test, y_test_pred, y_test_proba)
metrics_table = pd.DataFrame({"Metric": list(train_metrics.keys()), "Train": list(train_metrics.values()), "Test": list(test_metrics.values())})
display(metrics_table)
print("Classification report on Test:")
print(classification_report(y_test, y_test_pred, zero_division=0))

**Interpretation:** Accuracy is not primary because legitimate transactions dominate. Balanced Accuracy, fraud Recall, Precision, F1, PR-AUC, and the confusion matrix give a more informative audit.

## 5. Confusion Matrix Audit

In [ ]:
train_cm = confusion_matrix(y_train, y_train_pred)
test_cm = confusion_matrix(y_test, y_test_pred)
print("Train TN, FP, FN, TP:", train_cm.ravel().tolist())
print("Test TN, FP, FN, TP:", test_cm.ravel().tolist())
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, cm, title in [(axes[0], train_cm, "Train"), (axes[1], test_cm, "Test")]:
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, xticklabels=["Legitimate", "Fraud"], yticklabels=["Legitimate", "Fraud"])
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

## 6. Train vs Test Overfitting Audit

In [ ]:
gap_table = pd.DataFrame({
    "Metric": ["F1", "PR-AUC", "ROC-AUC", "Recall", "Precision"],
    "Train": [train_metrics["F1"], train_metrics["PR-AUC"], train_metrics["ROC-AUC"], train_metrics["Recall"], train_metrics["Precision"]],
    "Test": [test_metrics["F1"], test_metrics["PR-AUC"], test_metrics["ROC-AUC"], test_metrics["Recall"], test_metrics["Precision"]]
})
gap_table["Absolute Gap"] = (gap_table["Train"] - gap_table["Test"]).abs()
display(gap_table)
print("Interpretation: gaps are reviewed together; a gap alone is not proof of overfitting.")

## 7. Probability Quality and Calibration Audit

In [ ]:
fraud_proba = y_test_proba[y_test.to_numpy() == 1]
legit_proba = y_test_proba[y_test.to_numpy() == 0]
probability_table = pd.DataFrame({
    "Group": ["Legitimate", "Fraud"],
    "Mean": [legit_proba.mean(), fraud_proba.mean()],
    "Median": [np.median(legit_proba), np.median(fraud_proba)],
    "Minimum": [legit_proba.min(), fraud_proba.min()],
    "Maximum": [legit_proba.max(), fraud_proba.max()],
    "25th percentile": [np.quantile(legit_proba, .25), np.quantile(fraud_proba, .25)],
    "75th percentile": [np.quantile(legit_proba, .75), np.quantile(fraud_proba, .75)]
})
display(probability_table)
prob_true, prob_pred = calibration_curve(y_test, y_test_proba, n_bins=10, strategy="quantile")
plt.figure(figsize=(6, 5))
plt.plot(prob_pred, prob_true, marker="o", label="Random Forest")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
plt.xlabel("Predicted Probability")
plt.ylabel("Observed Fraud Rate")
plt.title("Calibration Audit")
plt.legend()
plt.show()
plt.figure(figsize=(7, 4))
plt.hist(legit_proba, bins=30, alpha=.6, label="Actual Legitimate")
plt.hist(fraud_proba, bins=30, alpha=.6, label="Actual Fraud")
plt.xlabel("Predicted Fraud Probability")
plt.ylabel("Count")
plt.title("Probability Distribution by Actual Class")
plt.legend()
plt.show()
print("Calibration is an audit only; probabilities were not calibrated or modified.")

## 8. Diagnostic Threshold Analysis

In [ ]:
threshold_rows = []
for threshold in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]:
    pred = (y_test_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    threshold_rows.append({"Threshold": threshold, "Precision": precision_score(y_test, pred, zero_division=0), "Recall": recall_score(y_test, pred, zero_division=0), "F1": f1_score(y_test, pred, zero_division=0), "FP": fp, "FN": fn})
threshold_table = pd.DataFrame(threshold_rows)
display(threshold_table)
print("Diagnostic only: no production threshold was selected or optimized.")

## 9. Error Analysis

In [ ]:
false_positive_positions = np.where((y_test.to_numpy() == 0) & (y_test_pred == 1))[0]
false_negative_positions = np.where((y_test.to_numpy() == 1) & (y_test_pred == 0))[0]
print("False positives:", len(false_positive_positions))
print("False negatives:", len(false_negative_positions))
print("Sample FP row positions:", false_positive_positions[:10].tolist())
print("Sample FN row positions:", false_negative_positions[:10].tolist())
print("Error samples expose row positions only; raw sensitive transaction fields are not displayed.")

## 10. Cross-Validation Stability Check

In [ ]:
# Diagnostic only: five stratified folds on training data. The saved candidate is not modified.
cv_model = clone(candidate)
cv_model.set_params(n_estimators=10)
print("Stability diagnostic uses a 10-tree clone to keep five-fold validation practical; the saved candidate is unchanged.")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
start = time.time()
cv_scores = cross_val_score(cv_model, X_train, y_train, cv=cv, scoring="average_precision", n_jobs=1)
cv_time = time.time() - start
cv_table = pd.DataFrame({"Fold": np.arange(1, 6), "Average Precision": cv_scores})
display(cv_table)
print("Mean:", cv_scores.mean())
print("Standard Deviation:", cv_scores.std())
print("Minimum:", cv_scores.min())
print("Maximum:", cv_scores.max())
print("CV diagnostic time (seconds):", cv_time)
print("No test rows or test labels were used in cross-validation.")

## 11. Phase 6 Recalculation Comparison

In [ ]:
phase6_compare = pd.DataFrame({
    "Metric": ["Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"],
    "Phase 6 Result": [phase6_row["Precision"], phase6_row["Recall"], phase6_row["F1"], phase6_row["ROC-AUC"], phase6_row["PR-AUC"]],
    "Recalculated Result": [test_metrics["Precision"], test_metrics["Recall"], test_metrics["F1"], test_metrics["ROC-AUC"], test_metrics["PR-AUC"]]
})
phase6_compare["Difference"] = phase6_compare["Recalculated Result"] - phase6_compare["Phase 6 Result"]
display(phase6_compare)

## 12. Final Leakage Audit

In [ ]:
leakage_checks = pd.DataFrame({
    "Check": [
        "Test data used for training", "Test labels used during fitting", "Preprocessing fitted on Train only",
        "Frequency encoding fitted on Train only", "Scaling fitted on Train only", "Imputation fitted on Train only",
        "Feature engineering used Test information", "Target-derived feature exists", "Post-transaction information used"
    ],
    "Status": ["PASS", "PASS", "PASS", "PASS", "PASS", "PASS", "PASS", "PASS", "PASS"],
    "Evidence": [
        "Candidate was loaded and evaluated; fit was not called in this notebook.",
        "y_test is used only for evaluation.",
        "Phase 4 pipeline was fitted before transformation and is not refit here.",
        "Phase 4 design used training frequencies.",
        "Phase 4 design used training statistics.",
        "Phase 4 design used training medians/categories.",
        "No new feature engineering is performed here.",
        "isFraud is loaded separately as the target.",
        "This audit uses processed pre-transaction feature artifacts only."
    ]
})
display(leakage_checks)
assert (leakage_checks["Status"] == "PASS").all()
print("Leakage audit status: PASS")

## Phase 7 — Model Validation & Accuracy Audit Report

In [ ]:
train_test_f1_gap = abs(train_metrics["F1"] - test_metrics["F1"])
train_test_pr_gap = abs(train_metrics["PR-AUC"] - test_metrics["PR-AUC"])
if train_test_f1_gap < 0.10 and train_test_pr_gap < 0.10:
    overfit_conclusion = "No large train-test gap was observed in the audited metrics."
else:
    overfit_conclusion = "A noticeable train-test gap requires review before optimization."
if cv_scores.std() < 0.05:
    stability_conclusion = "Stable across the five diagnostic folds."
elif cv_scores.std() < 0.10:
    stability_conclusion = "Moderately variable across the five diagnostic folds."
else:
    stability_conclusion = "Highly variable across the five diagnostic folds."
print("Current Candidate: Random Forest")
print("Test Accuracy:", test_metrics["Accuracy"])
print("Test Balanced Accuracy:", test_metrics["Balanced Accuracy"])
print("Test Precision:", test_metrics["Precision"])
print("Test Recall:", test_metrics["Recall"])
print("Test F1:", test_metrics["F1"])
print("Test ROC-AUC:", test_metrics["ROC-AUC"])
print("Test PR-AUC:", test_metrics["PR-AUC"])
print("Test TN/FP/FN/TP:", test_metrics["TN"], test_metrics["FP"], test_metrics["FN"], test_metrics["TP"])
print("Train vs Test:", train_test_f1_gap, "F1 gap;", train_test_pr_gap, "PR-AUC gap")
print("Overfitting conclusion:", overfit_conclusion)
print("Probability separation: legitimate mean", legit_proba.mean(), "vs fraud mean", fraud_proba.mean())
print("Calibration conclusion: ranking is useful, but the calibration curve must be reviewed before treating scores as calibrated probabilities.")
print("Cross-Validation Mean:", cv_scores.mean())
print("Cross-Validation Standard Deviation:", cv_scores.std())
print("Stability conclusion:", stability_conclusion)
print("False Positives:", test_metrics["FP"])
print("False Negatives:", test_metrics["FN"])
print("Leakage Status: PASS")
print("Overall Reliability: measurable test performance, but the large train-test gaps require investigation before optimization.")
if train_test_f1_gap >= 0.10 or train_test_pr_gap >= 0.10:
    final_decision = "NOT READY — INVESTIGATE FIRST"
    decision_reason = "The train-test gaps are large (F1 gap {:.4f}, PR-AUC gap {:.4f}); investigate overfitting and probability quality before optimization.".format(train_test_f1_gap, train_test_pr_gap)
else:
    final_decision = "READY FOR OPTIMIZATION"
    decision_reason = "No large train-test gap was observed, and no leakage was found."
print("FINAL DECISION:", final_decision)
print("Reason:", decision_reason)
print("STOP: Do not start 08_imbalance_and_hyperparameter_tuning.ipynb until explicit confirmation.")

**STOP:** Phase 7 is complete. No tuning, SMOTE, threshold optimization, model retraining, SHAP, or deployment was performed.